In [0]:
from pyspark.sql.functions import *

In [0]:
df_silver = spark.table('depi.silver_sales')

In [0]:
df_silver.display()

Product_ID,Sale_Date,Sales_Rep,Region,Sales_Amount,Quantity_Sold,Product_Category,Unit_Cost,Unit_Price,Customer_Type,Discount,Payment_Method,Sales_Channel,DataSource,Revenue,Profit,Discount_Amount,Sale_Month,Sale_Year,Day_of_Week
1052,2023-02-03,Bob,North,5053.97,18,Furniture,152.75,267.22,Returning,0.09,Cash,Online,Sales_data,4809.96,2060.46,454.86,2,2023,6
1093,2023-04-21,Bob,West,4384.02,17,Furniture,3816.39,4209.44,Returning,0.11,Unknown,Retail,Sales_data,71560.48,6681.85,482.24,4,2023,6
1015,2023-09-21,Unknown,South,4631.23,30,Food,261.56,371.4,Returning,0.2,Bank Transfer,Retail,Sales_data,11142.0,3295.2,926.25,9,2023,5
1072,2023-08-24,Bob,South,2167.94,39,Clothing,4330.03,4467.75,New,0.02,Credit Card,Retail,Sales_data,174242.25,5371.08,43.36,8,2023,5
1061,2023-03-24,Charlie,East,3750.2,13,Electronics,637.37,692.71,New,0.08,Credit Card,Online,Sales_data,9005.23,719.42,300.02,3,2023,6
1021,2023-02-11,Charlie,West,3761.15,32,Food,900.79,1106.51,New,0.21,Cash,Online,Sales_data,35408.32,6583.04,789.84,2,2023,7
1083,2023-04-11,Bob,West,618.31,29,Furniture,2408.81,2624.09,Returning,0.14,Cash,Online,Sales_data,76098.61,6243.12,86.56,4,2023,3
1087,2023-01-06,Eve,South,7698.92,46,Furniture,3702.51,3964.65,New,0.12,Bank Transfer,Online,Sales_data,182373.9,12058.44,923.87,1,2023,6
1075,2023-06-29,David,South,4223.39,30,Furniture,738.06,1095.4499999999998,New,0.05,Bank Transfer,Online,Sales_data,32863.5,10721.7,211.17,6,2023,5
1075,2023-10-09,Charlie,West,8239.58,18,Clothing,2228.35,2682.34,New,0.13,Bank Transfer,Online,Sales_data,48282.12,8171.82,1071.15,10,2023,2


In [0]:
from pyspark.sql.functions import sum, round, avg

gold_by_region = df_silver.groupBy("Region") \
    .agg(
        round(sum("Revenue"), 2).alias("Total_Revenue"),
        round(sum("Profit"), 2).alias("Total_Profit"),
        sum("Quantity_Sold").alias("Total_Quantity"),
        round(avg("Discount"), 2).alias("Avg_Discount")
    ).orderBy("Total_Revenue", ascending=False)

gold_by_region.createOrReplaceTempView("gold_sales_by_region")
display(gold_by_region)

Region,Total_Revenue,Total_Profit,Total_Quantity,Avg_Discount
North,1.820840385E7,1661461.2,6705,0.15
East,1.807740105E7,1650557.2,6356,0.16
West,1.776156865E7,1656091.77,6486,0.15
South,1.628256716E7,1519736.9,5808,0.15


In [0]:
gold_by_category = df_silver.groupBy("Product_Category") \
    .agg(
        round(sum("Revenue"), 2).alias("Total_Revenue"),
        round(sum("Profit"), 2).alias("Total_Profit"),
        sum("Quantity_Sold").alias("Total_Quantity"),
        round(avg("Discount"), 2).alias("Avg_Discount")
    ).orderBy("Total_Revenue", ascending=False)

gold_by_category.createOrReplaceTempView("gold_sales_by_category")
display(gold_by_category)

Product_Category,Total_Revenue,Total_Profit,Total_Quantity,Avg_Discount
Clothing,1.928673393E7,1712957.8,6922,0.16
Furniture,1.833028056E7,1779461.16,6729,0.16
Electronics,1.757104057E7,1574320.06,6096,0.14
Food,1.514188565E7,1421108.05,5608,0.15


In [0]:
from pyspark.sql.functions import sum, round, count

gold_by_time = df_silver.groupBy("Sale_Year", "Sale_Month") \
    .agg(
        round(sum("Revenue"), 2).alias("Total_Revenue"),
        round(sum("Profit"), 2).alias("Total_Profit"),
        count("*").alias("Total_Transactions")
    ).orderBy("Sale_Year", "Sale_Month")

gold_by_time.write.mode("overwrite").format("delta").saveAsTable("workspace.depi.gold_sales_by_time")
display(gold_by_time)

Sale_Year,Sale_Month,Total_Revenue,Total_Profit,Total_Transactions
2023,1,7427821.05,655202.59,100
2023,2,5627477.45,478053.08,75
2023,3,6008259.41,528631.81,80
2023,4,5468497.62,496006.82,81
2023,5,5800626.51,489472.15,72
2023,6,6142561.23,605992.2,92
2023,7,4751432.42,447678.76,68
2023,8,6716190.61,558417.69,93
2023,9,4557015.58,472940.52,68
2023,10,6560606.39,584679.83,88


In [0]:
gold_by_payment = df_silver.groupBy("Payment_Method") \
    .agg(
        round(sum("Revenue"), 2).alias("Total_Revenue"),
        count("*").alias("Total_Transactions")
    ).orderBy("Total_Revenue", ascending=False)

gold_by_payment.write.mode("overwrite").format("delta").saveAsTable("workspace.depi.gold_sales_by_payment")
display(gold_by_payment)

Payment_Method,Total_Revenue,Total_Transactions
Bank Transfer,2.220440548E7,312
Credit Card,2.051546569E7,306
Cash,2.036424186E7,284
Unknown,7245827.68,98


In [0]:
from pyspark.sql.functions import max

gold_kpi = df_silver.agg(
    round(sum("Revenue"), 2).alias("Total_Revenue"),
    count("*").alias("Total_Transactions"),
    round(sum("Profit"), 2).alias("Total_Profit")
)

gold_kpi.write.mode("overwrite").format("delta").saveAsTable("workspace.depi.gold_kpi_summary")
display(gold_kpi)

Total_Revenue,Total_Transactions,Total_Profit
7.032994071E7,1000,6487847.07


In [0]:
gold_by_region.write.mode("overwrite").format("delta").saveAsTable("workspace.depi.gold_sales_by_region")
gold_by_category.write.mode("overwrite").format("delta").saveAsTable("workspace.depi.gold_sales_by_category")

print("Gold tables created successfully!")

Gold tables created successfully!
